In [1]:
from typing import Optional, Literal
from pydantic import BaseModel, Field
from openai import OpenAI
import os
import logging

In [2]:
logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s - %(levelname)s - %(message)s",
    datefmt="%Y-%m-%d %H:%M:%S",
)
logger = logging.getLogger(__name__)

In [3]:
client = OpenAI(api_key=os.getenv("OPENAI_API_KEY"))
model = "gpt-5-nano"

In [4]:
class CalendarRequestType(BaseModel):
    """Router LLM call: Determine the type of calendar request"""

    request_type: Literal["new_event", "modify_event", "other"] = Field(
        description="Type of calendar request being made"
    )
    confidence_score: float = Field(description="Confidence score between 0 and 1")
    description: str = Field(description="Cleaned description of the request")


class NewEventDetails(BaseModel):
    """Details for creating a new event"""

    name: str = Field(description="Name of the event")
    date: str = Field(description="Date and time of the event (ISO 8601)")
    duration_minutes: int = Field(description="Duration in minutes")
    participants: list[str] = Field(description="List of participants")


class Change(BaseModel):
    """Details for changing an existing event"""

    field: str = Field(description="Field to change")
    new_value: str = Field(description="New value for the field")


class ModifyEventDetails(BaseModel):
    """Details for modifying an existing event"""

    event_identifier: str = Field(
        description="Description to identify the existing event"
    )
    changes: list[Change] = Field(description="List of changes to make")
    participants_to_add: list[str] = Field(description="New participants to add")
    participants_to_remove: list[str] = Field(description="Participants to remove")


class CalendarResponse(BaseModel):
    """Final response format"""

    success: bool = Field(description="Whether the operation was successful")
    message: str = Field(description="User-friendly response message")
    calendar_link: Optional[str] = Field(description="Calendar link if applicable")

In [ ]:
def route_calendar_request(user_input: str) -> CalendarRequestType:
    """Router LLM call to determine the type of calendar request"""
    logger.info("Routing calendar request")

    completion = client.beta.chat.completions.parse(
        model=model,
        messages=[
            {
                "role": "system",
                "content": "Determine if this is a request to create a new calendar event or modify an existing one.",
            },
            {"role": "user", "content": user_input},
        ],
        response_format=CalendarRequestType,
    )
    result = completion.choices[0].message.parsed
    print(("\n Result from route_calendar_request"))
    print(result.model_dump())
    print("\n")
    
    logger.info(
        f"Request routed as: {result.request_type} with confidence: {result.confidence_score}"
    )
    return result

In [6]:
def handle_new_event(description: str) -> CalendarResponse:
    """Process a new event request"""
    logger.info("Processing new event request")

    # Get event details
    completion = client.beta.chat.completions.parse(
        model=model,
        messages=[
            {
                "role": "system",
                "content": "Extract details for creating a new calendar event.",
            },
            {"role": "user", "content": description},
        ],
        response_format=NewEventDetails,
    )
    details = completion.choices[0].message.parsed
    
    print(("\n Result from handle_new_event"))
    print(details.model_dump())
    print("\n")

    logger.info(f"New event: {details.model_dump_json(indent=2)}")

    # Generate response
    return CalendarResponse(
        success=True,
        message=f"Created new event '{details.name}' for {details.date} with {', '.join(details.participants)}",
        calendar_link=f"calendar://new?event={details.name}",
    )

In [7]:
def handle_modify_event(description: str) -> CalendarResponse:
    """Process an event modification request"""
    logger.info("Processing event modification request")

    # Get modification details
    completion = client.beta.chat.completions.parse(
        model=model,
        messages=[
            {
                "role": "system",
                "content": "Extract details for modifying an existing calendar event.",
            },
            {"role": "user", "content": description},
        ],
        response_format=ModifyEventDetails,
    )
    details = completion.choices[0].message.parsed
    
    print(("\n Result from handle_modify_event"))
    print(details.model_dump())
    print("\n")

    logger.info(f"Modified event: {details.model_dump_json(indent=2)}")

    # Generate response
    return CalendarResponse(
        success=True,
        message=f"Modified event '{details.event_identifier}' with the requested changes",
        calendar_link=f"calendar://modify?event={details.event_identifier}",
    )

In [8]:
def process_calendar_request(user_input: str) -> Optional[CalendarResponse]:
    """Main function implementing the routing workflow"""
    logger.info("Processing calendar request")

    # Route the request
    route_result = route_calendar_request(user_input)
    
    print(("\n Result of value returned from the route_calendar_request"))
    print(route_result.model_dump())
    print("\n")

    # Check confidence threshold
    if route_result.confidence_score < 0.7:
        logger.warning(f"Low confidence score: {route_result.confidence_score}")
        return None

    # Route to appropriate handler
    if route_result.request_type == "new_event":
        return handle_new_event(route_result.description)
    elif route_result.request_type == "modify_event":
        return handle_modify_event(route_result.description)
    else:
        logger.warning("Request type not supported")
        return None

In [11]:
#Test with new event : 

new_event_input = "Let's schedule a team meeting next Tuesday at 2pm with Alice and Bob"
result = process_calendar_request(new_event_input)
if result:
    print(f"Response: {result.model_dump()}")

2025-10-13 16:45:00 - INFO - Processing calendar request
2025-10-13 16:45:00 - INFO - Routing calendar request
2025-10-13 16:45:09 - INFO - HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"
2025-10-13 16:45:09 - INFO - Request routed as: new_event with confidence: 0.92
2025-10-13 16:45:09 - INFO - Processing new event request



 Result from route_calendar_request
{'request_type': 'new_event', 'confidence_score': 0.92, 'description': 'Create a new calendar event: team meeting next Tuesday at 2:00 PM with Alice and Bob.'}



 Result of value returned from the route_calendar_request
{'request_type': 'new_event', 'confidence_score': 0.92, 'description': 'Create a new calendar event: team meeting next Tuesday at 2:00 PM with Alice and Bob.'}




2025-10-13 16:45:23 - INFO - HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"
2025-10-13 16:45:23 - INFO - New event: {
  "name": "Team meeting",
  "date": "2025-10-14T14:00:00",
  "duration_minutes": 60,
  "participants": [
    "Alice",
    "Bob"
  ]
}



 Result from handle_new_event
{'name': 'Team meeting', 'date': '2025-10-14T14:00:00', 'duration_minutes': 60, 'participants': ['Alice', 'Bob']}


Response: {'success': True, 'message': "Created new event 'Team meeting' for 2025-10-14T14:00:00 with Alice, Bob", 'calendar_link': 'calendar://new?event=Team meeting'}


In [10]:
invalid_input = "What's the weather like today?"
result = process_calendar_request(invalid_input)
if not result:
    print("Request not recognized as a calendar operation")

2025-10-13 16:40:57 - INFO - Processing calendar request
2025-10-13 16:40:57 - INFO - Routing calendar request
2025-10-13 16:41:02 - INFO - HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"
2025-10-13 16:41:02 - INFO - Request routed as: other with confidence: 0.92
2025-10-13 16:41:02 - WARNING - Request type not supported



 Result from route_calendar_request
{'request_type': 'other', 'confidence_score': 0.92, 'description': 'User asked for current weather conditions for today.'}



 Result of value returned from the route_calendar_request
{'request_type': 'other', 'confidence_score': 0.92, 'description': 'User asked for current weather conditions for today.'}


Request not recognized as a calendar operation


In [ ]:
#It gets the word other from the Pydantic validation of CalendarRequestType